 - Load the dataset
 - Calculate datetime by combining Date and Time columns
 - Use only Open value in the candlestick. We just need one representative value for a minute. So, drop other columns in place.

In [1]:
import pandas as pd

n50 = pd.read_csv("dataset/nifty50_candlestick_data.csv")
n50["datetime"] = pd.to_datetime(n50["Date"] + " " + n50["Time"], format="%d-%m-%Y %H:%M:%S")
n50.set_index("datetime", inplace=True)
n50.drop(columns=["Date", "Time", "High", "Low", "Close", "Instrument"], inplace=True)


In [2]:
print(n50.head())
print("\nShape: ", n50.shape)

                        Open
datetime                    
2015-01-09 09:15:00  8285.45
2015-01-09 09:16:00  8292.60
2015-01-09 09:17:00  8287.40
2015-01-09 09:18:00  8294.25
2015-01-09 09:19:00  8300.60

Shape:  (852087, 1)


Let's include only the time when Market is open. For some reason the market could've been open after hours. Or there could be a glitch in the data. In either case, we're ignoring those values.

In [3]:
market_hours_filter = (n50.index.time >= pd.to_datetime("09:15:00").time()) & (n50.index.time <= pd.to_datetime("15:30:00").time())
n50 = n50[market_hours_filter]
print("\nShape after filtering market hours: ", n50.shape)


Shape after filtering market hours:  (851460, 1)


Convert the data into a time-series data. Group all Open value that are corresponding to one day into a single row. The column value should be time in 1 min interval.

There could be multiple open values for a particular minute of a day. If the candle-stick has collected second level-data, we would have 60 Opens for one minute. As per the description of the data-set, we expect only one value per minute. Still we expect some glitches in the huge dataset. So, we use aggregate function `aggfunc` which takes the first value. *We have the option to choose a median or average value in that minute*.

In [4]:
n50['date'] = n50.index.date
n50['time'] = n50.index.strftime('%H:%M')

n50_pivot = n50.pivot_table(index='date', columns='time', values='Open', aggfunc='first')

In [5]:
n50 = n50_pivot.copy()
print(n50.head())

time          09:15    09:16    09:17    09:18   09:19    09:20    09:21  \
date                                                                       
2015-01-09  8285.45  8292.60  8287.40  8294.25  8300.6  8300.50  8300.65   
2015-01-12  8291.35  8254.20  8255.25  8258.15  8263.2  8267.45  8266.05   
2015-01-13  8346.15  8355.15  8348.70  8344.50  8342.5  8340.35  8339.75   
2015-01-14  8307.25  8300.85  8307.00  8309.05  8305.4  8304.70  8302.20   
2015-01-15  8425.20  8440.45  8394.35  8386.05  8401.1  8428.00  8408.25   

time          09:22    09:23    09:24  ...   15:20    15:21    15:22    15:23  \
date                                   ...                                      
2015-01-09  8302.45  8294.85  8295.20  ...  8280.8  8282.35  8283.40  8284.35   
2015-01-12  8268.80  8273.85  8266.75  ...  8329.5  8326.55  8328.05  8328.05   
2015-01-13  8340.45  8333.30  8326.05  ...  8304.9  8305.75  8306.50  8307.15   
2015-01-14  8293.10  8296.70  8306.85  ...  8280.1  8278.90  8

Let's drop a day if any of the column missing in it

In [6]:
print(n50.shape)

missing_days = n50.isnull().any(axis=1)
print("\nNumber of days with missing data: ", missing_days.sum())

n50 = n50[~missing_days]

print(n50.shape)

(2273, 375)

Number of days with missing data:  30
(2243, 375)


We're going to convert it into a actual regression dataset. The `X` will be Open price from 9.15 AM to 2.30 PM. The `y` will be the median value of the Open prices between 2.31 PM and 3.30 PM.

*The idea is to predict the median value of the next one hour market. If the predicted median value is higher than current market price, we'll buy the index or we'll sell the index.*

In [7]:
X = n50.loc[:, '09:15':'14:30'].values
y = n50.loc[:, '14:31':'15:30'].median(axis=1).values

print("\nX head: ", X[:5])
print("y head: ", y[:5])

n50 = pd.DataFrame(X, index=n50.index, columns=n50.columns[:X.shape[1]])
n50['target'] = y

print(n50.head())



X head:  [[8285.45 8292.6  8287.4  ... 8237.65 8239.05 8240.  ]
 [8291.35 8254.2  8255.25 ... 8315.15 8317.15 8319.3 ]
 [8346.15 8355.15 8348.7  ... 8329.45 8325.85 8317.55]
 [8307.25 8300.85 8307.   ... 8283.5  8285.55 8285.25]
 [8425.2  8440.45 8394.35 ... 8487.6  8489.15 8491.6 ]]
y head:  [8272.55 8317.45 8298.25 8272.45 8508.75]
time          09:15    09:16    09:17    09:18   09:19    09:20    09:21  \
date                                                                       
2015-01-09  8285.45  8292.60  8287.40  8294.25  8300.6  8300.50  8300.65   
2015-01-12  8291.35  8254.20  8255.25  8258.15  8263.2  8267.45  8266.05   
2015-01-13  8346.15  8355.15  8348.70  8344.50  8342.5  8340.35  8339.75   
2015-01-14  8307.25  8300.85  8307.00  8309.05  8305.4  8304.70  8302.20   
2015-01-15  8425.20  8440.45  8394.35  8386.05  8401.1  8428.00  8408.25   

time          09:22    09:23    09:24  ...    14:22    14:23    14:24  \
date                                   ...               